# 02 · Depth regressionAnalytic Pearson residuals → `.X` (SenePy input only)Each block is **Why → Test → Display**. Code is lifted from `00.5_regressionCopy4.ipynb` with the source cell number recorded; anything not traceable to source is marked `⟨NEW⟩`.

## Configuration**Why.** One config module resolves every path and parameter from a single dataset key, so the only thing that differs between cohorts is that key. Merged from the dataset dicts already present in the source notebooks.

In [ ]:
from pathlib import Pathimport warnings; warnings.filterwarnings('ignore')from config import CFG, assert_layersimport config as CDATASET = 'psychad_aging'      # <<< the only line you changecfg = CFG.for_dataset(DATASET)cfg.echo()def why(block, question, rationale=None):    """Print the Why so it lands in executed output, not only in markdown."""    print("\n" + "=" * 78)    print(f"  {block}")    print("=" * 78)    print(f"  Q: {question}")    if rationale:        for line in rationale.split(" | "):            print(f"     {line}")    print()def gate(label, ok, detail=""):    """Fail loudly. A failed gate stops the module rather than flowing downstream."""    mark = "OK  " if ok else "FAIL"    print(f"  [{mark}] {label}{'  — ' + detail if detail else ''}")    if not ok:        raise AssertionError(f"GATE FAILED: {label}. {detail}")    return ok

## Setup**Why.** Requires scanpy ≥ 1.9 for `experimental.pp.normalize_pearson_residuals`.<sub>source: `00.5_regressionCopy4.ipynb` cell 2</sub>

In [ ]:
why("Setup", "Requires scanpy ≥ 1")

In [ ]:
# ── source: 00.5_regressionCopy4.ipynb cell 2 ──import scanpy as scimport numpy as npimport scipy.sparse as spfrom scipy.stats import spearmanrfrom pathlib import Pathimport timeimport warningswarnings.filterwarnings('ignore')print(f"scanpy: {sc.__version__}")print(f"numpy: {np.__version__}")

## Load Data**Why.** Asserts `layers['counts']` exists and holds integers — residuals need raw counts.<sub>source: `00.5_regressionCopy4.ipynb` cell 6</sub>

In [ ]:
why("Load Data", "Asserts `layers['counts']` exists and holds integers — residuals need raw counts")

In [ ]:
# ── source: 00.5_regressionCopy4.ipynb cell 6 ──print("\n" + "="*80)print("LOADING DATA")print("="*80)start_time = time.time()adata = sc.read_h5ad(INPUT_FILE)print(f"\n  Loaded: {INPUT_FILE.name}")print(f"  Cells: {adata.n_obs:,}")print(f"  Genes: {adata.n_vars:,}")print(f"  .X dtype: {adata.X.dtype}")print(f"  .X sparse: {sp.issparse(adata.X)}")print(f"  Layers: {list(adata.layers.keys())}")# Verify raw counts existif 'counts' not in adata.layers:    raise ValueError("layers['counts'] not found! Module 00 must save raw counts.")# Verify .X is log-normalizedif sp.issparse(adata.X):    x_min = float(adata.X.data.min()) if len(adata.X.data) > 0 else 0    x_max = float(adata.X.data.max()) if len(adata.X.data) > 0 else 0else:    x_min = float(np.min(adata.X))    x_max = float(np.max(adata.X))print(f"\n  .X range: [{x_min:.3f}, {x_max:.3f}]")print(f"  (Expected: 0 to ~10 for log-normalized data)")# Verify raw counts are integersif sp.issparse(adata.layers['counts']):    c_max = float(adata.layers['counts'].data.max()) if len(adata.layers['counts'].data) > 0 else 0    sample_data = np.array(adata.layers['counts'].data[:1000])else:    c_max = float(np.max(adata.layers['counts']))    sample_data = np.array(adata.layers['counts'].flat[:1000])is_integer = np.allclose(sample_data, np.round(sample_data))print(f"\n  layers['counts'] max: {c_max:.0f}")print(f"  Contains integers: {is_integer}")print(f"\n✓ Loaded in {time.time() - start_time:.1f}s")

## Pre-Correction UMI Confounding Check**Why.** Baseline depth coupling, recorded before correction so the effect is measurable rather than assumed.<sub>source: `00.5_regressionCopy4.ipynb` cell 8</sub>

In [ ]:
why("Pre-Correction UMI Confounding Check", "Baseline depth coupling, recorded before correction so the effect is measurable rather than assumed")

In [ ]:
# ── source: 00.5_regressionCopy4.ipynb cell 8 ──print("\n" + "="*80)print("PRE-CORRECTION UMI CONFOUNDING CHECK")print("="*80)np.random.seed(42)idx = np.random.choice(adata.n_obs, min(5000, adata.n_obs), replace=False)X_sample = adata.X[idx]if sp.issparse(X_sample):    mean_expr_pre = np.asarray(X_sample.mean(axis=1)).flatten()elif isinstance(X_sample, np.ndarray):    mean_expr_pre = X_sample.mean(axis=1).flatten()else:    mean_expr_pre = np.asarray(X_sample).mean(axis=1).flatten()total_counts = adata.obs['total_counts'].iloc[idx].valuesrho_pre, pval_pre = spearmanr(mean_expr_pre, total_counts)print(f"\n  Spearman ρ (mean .X vs total_counts): {rho_pre:.4f} (p={pval_pre:.2e})")if abs(rho_pre) > 0.8:    print(f"  ⚠ Strong UMI confounding (ρ={rho_pre:.2f})")    print(f"    → Pearson residuals will correct this")elif abs(rho_pre) > 0.5:    print(f"  ~ Moderate UMI confounding (ρ={rho_pre:.2f})")else:    print(f"  ✓ Low UMI confounding (ρ={rho_pre:.2f})")print(f"\n✓ Baseline recorded")

## Prepare for Pearson Residuals**Why.** Stores log-normalized into `layers['lognorm']` and sets `.X` to raw counts, which is what the residual computation expects as input.<sub>source: `00.5_regressionCopy4.ipynb` cell 10</sub>

In [ ]:
why("Prepare for Pearson Residuals", "Stores log-normalized into `layers['lognorm']` and sets `")

In [ ]:
# ── source: 00.5_regressionCopy4.ipynb cell 10 ──print("\n" + "="*80)print("PREPARING FOR PEARSON RESIDUALS")print("="*80)# Save log-normalized as layerprint("\n  → Saving log-normalized .X to layers['lognorm']...")adata.layers['lognorm'] = adata.X.copy()print("    ✓ layers['lognorm'] stored")# Set .X to raw countsprint("\n  → Setting .X = raw counts from layers['counts']...")adata.X = adata.layers['counts'].copy()# Clear log1p flagif 'log1p' in adata.uns:    del adata.uns['log1p']print(f"\n  Data layers:")print(f"    .X          → raw counts (input for Pearson residuals)")print(f"    ['counts']  → raw counts (backup)")print(f"    ['lognorm'] → log-normalized (for visualization)")print(f"\n✓ Ready for Pearson residuals")

## Compute Pearson Residuals**Why.** Clipped at ±30 (SCTransform v2 convention) so rare lowly-expressed genes don't dominate. Memory-intensive: residuals densify.<sub>source: `00.5_regressionCopy4.ipynb` cell 12</sub>

In [ ]:
why("Compute Pearson Residuals", "Clipped at ±30 (SCTransform v2 convention) so rare lowly-expressed genes don't dominate")

In [ ]:
# ── source: 00.5_regressionCopy4.ipynb cell 12 ──print("\n" + "="*80)print("COMPUTING PEARSON RESIDUALS")print("="*80)print(f"\n  Model:     Negative binomial GLM")print(f"  Clipping:  ±{CLIP_VALUE}")print(f"  Cells:     {adata.n_obs:,}")print(f"  Genes:     {adata.n_vars:,}")print("\n  → Computing Pearson residuals...")start_time = time.time()sc.experimental.pp.normalize_pearson_residuals(adata, clip=CLIP_VALUE)elapsed = time.time() - start_timeprint(f"\n  ✓ Computed in {elapsed:.1f}s ({elapsed/60:.1f} min)")print(f"    .X shape: {adata.X.shape}")print(f"    .X dtype: {adata.X.dtype}")print(f"    .X sparse: {sp.issparse(adata.X)}")

## Post-Correction Verification**Why.** ⟨NEW⟩ Note on interpretation: per-gene expression-vs-depth is **not** the gate for this module, because residuals never reach a per-gene analysis. The gate that matters is senescence-score-vs-depth, and it is measured in module 03 after scoring.<sub>source: `00.5_regressionCopy4.ipynb` cell 14</sub>

In [ ]:
why("Post-Correction Verification", "⟨NEW⟩ Note on interpretation: per-gene expression-vs-depth is **not** the gate for this module, because residuals never reach a per-gene analysis")

In [ ]:
# ── source: 00.5_regressionCopy4.ipynb cell 14 ──print("\n" + "="*80)print("POST-CORRECTION VERIFICATION")print("="*80)# ── A. Overall correlation ────────────────────────────────────────────────────print("\nA. OVERALL CORRELATION")print(f"{'─'*60}")X_sample = adata.X[idx]if sp.issparse(X_sample):    mean_expr_post = np.asarray(X_sample.mean(axis=1)).flatten()elif isinstance(X_sample, np.ndarray):    mean_expr_post = X_sample.mean(axis=1).flatten()else:    mean_expr_post = np.asarray(X_sample).mean(axis=1).flatten()rho_post, pval_post = spearmanr(mean_expr_post, total_counts)print(f"\n  {'Metric':<35} {'Before':>12} {'After':>12}")print(f"  {'-'*61}")print(f"  {'Spearman ρ (mean expr vs UMI)':<35} {rho_pre:>12.4f} {rho_post:>12.4f}")print(f"  {'|ρ| improvement':<35} {'':>12} {abs(rho_pre) - abs(rho_post):>12.4f}")# ── B. Value range ────────────────────────────────────────────────────────────print(f"\nB. VALUE RANGE")print(f"{'─'*60}")if sp.issparse(adata.X):    x_min = float(adata.X.data.min()) if len(adata.X.data) > 0 else 0    x_max = float(adata.X.data.max()) if len(adata.X.data) > 0 else 0else:    x_min = float(np.min(adata.X))    x_max = float(np.max(adata.X))print(f"\n  .X range: [{x_min:.3f}, {x_max:.3f}]")print(f"  Has negative values: {x_min < 0} (expected: True)")print(f"  Clipped to ±{CLIP_VALUE}: {x_max <= CLIP_VALUE and x_min >= -CLIP_VALUE}")# ── C. Per-gene correlation ───────────────────────────────────────────────────print(f"\nC. PER-GENE CORRELATION (100 random genes)")print(f"{'─'*60}")np.random.seed(42)gene_idx = np.random.choice(adata.n_vars, 100, replace=False)rhos_pearson = []rhos_lognorm = []for g in gene_idx:    if sp.issparse(adata.X):        gene_expr = np.asarray(adata.X[:, g].todense()).flatten()    else:        gene_expr = adata.X[:, g].flatten()        if sp.issparse(adata.layers['lognorm']):        gene_ln = np.asarray(adata.layers['lognorm'][:, g].todense()).flatten()    else:        gene_ln = adata.layers['lognorm'][:, g].flatten()        tc = adata.obs['total_counts'].values    r1, _ = spearmanr(gene_expr, tc)    r2, _ = spearmanr(gene_ln, tc)    rhos_pearson.append(r1)    rhos_lognorm.append(r2)rhos_pearson = np.array(rhos_pearson)rhos_lognorm = np.array(rhos_lognorm)print(f"\n  {'Metric':<40} {'Log-norm':>10} {'Pearson':>10}")print(f"  {'-'*62}")print(f"  {'Mean |ρ| (gene vs UMI)':<40} {np.mean(np.abs(rhos_lognorm)):>10.4f} {np.mean(np.abs(rhos_pearson)):>10.4f}")print(f"  {'Median |ρ| (gene vs UMI)':<40} {np.median(np.abs(rhos_lognorm)):>10.4f} {np.median(np.abs(rhos_pearson)):>10.4f}")print(f"  {'Genes with |ρ| > 0.3':<40} {(np.abs(rhos_lognorm) > 0.3).sum():>10} {(np.abs(rhos_pearson) > 0.3).sum():>10}")print(f"  {'Genes with |ρ| > 0.1':<40} {(np.abs(rhos_lognorm) > 0.1).sum():>10} {(np.abs(rhos_pearson) > 0.1).sum():>10}")print(f"\n✓ Verification complete")

## Visualization**Why.** Before/after depth coupling, overall and per-gene.<sub>source: `00.5_regressionCopy4.ipynb` cell 16</sub>

In [ ]:
why("Visualization", "Before/after depth coupling, overall and per-gene")

In [ ]:
# ── source: 00.5_regressionCopy4.ipynb cell 16 ──import matplotlib.pyplot as pltprint("\n" + "="*80)print("VISUALIZATION")print("="*80)plt.rcParams.update({    'figure.dpi': 150, 'savefig.dpi': 300,    'font.size': 10, 'axes.labelsize': 10,    'axes.titlesize': 11, 'legend.fontsize': 9,    'font.family': 'sans-serif', 'axes.linewidth': 1.0,    'axes.grid': False, 'pdf.fonttype': 42,})# ── Before vs After scatter ───────────────────────────────────────────────────fig, axes = plt.subplots(1, 2, figsize=(10, 4))for ax in axes:    ax.spines['top'].set_visible(False)    ax.spines['right'].set_visible(False)ln_sample = adata.layers['lognorm'][idx]if sp.issparse(ln_sample):    ln_mean = np.asarray(ln_sample.mean(axis=1)).flatten()else:    ln_mean = ln_sample.mean(axis=1).flatten()axes[0].scatter(total_counts, ln_mean, s=2, alpha=0.2, c='#4E79A7', rasterized=True)axes[0].set_xlabel('Total UMI Counts')axes[0].set_ylabel('Mean Expression')axes[0].set_title(f'Log-normalized (ρ = {rho_pre:.3f})')axes[1].scatter(total_counts, mean_expr_post, s=2, alpha=0.2, c='#E15759', rasterized=True)axes[1].set_xlabel('Total UMI Counts')axes[1].set_ylabel('Mean Pearson Residual')axes[1].set_title(f'Pearson Residuals (ρ = {rho_post:.3f})')plt.suptitle(f'{DATASET}: Effect of UMI Correction', fontsize=12, y=1.02)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_umi_correction_scatter.svg', dpi=300, bbox_inches='tight')plt.show()print(f"✓ Saved: {DATASET}_umi_correction_scatter.svg")# ── Per-gene rho distribution ─────────────────────────────────────────────────fig, ax = plt.subplots(figsize=(6, 4))ax.spines['top'].set_visible(False)ax.spines['right'].set_visible(False)ax.hist(rhos_lognorm, bins=30, alpha=0.6, color='#4E79A7', label='Log-normalized', edgecolor='white', linewidth=0.5)ax.hist(rhos_pearson, bins=30, alpha=0.6, color='#E15759', label='Pearson residuals', edgecolor='white', linewidth=0.5)ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)ax.set_xlabel('Spearman ρ (gene expression vs UMI)')ax.set_ylabel('Number of genes')ax.set_title(f'{DATASET}: Per-Gene UMI Correlation (n=100 genes)')ax.legend(frameon=False)plt.tight_layout()plt.savefig(FIGURES_DIR / f'{DATASET}_umi_correction_pergene.svg', dpi=300, bbox_inches='tight')plt.show()print(f"✓ Saved: {DATASET}_umi_correction_pergene.svg")

## Save Results**Why.** `.X` residuals · `layers['counts']` raw · `layers['lognorm']` log-normalized. This file is large and re-derivable from counts — treat it as disposable.<sub>source: `00.5_regressionCopy4.ipynb` cell 18</sub>

In [ ]:
why("Save Results", "`")

In [ ]:
# ── source: 00.5_regressionCopy4.ipynb cell 18 ──print("\n" + "="*80)print("SAVING")print("="*80)print(f"\n  Final data structure:")print(f"    Cells: {adata.n_obs:,}")print(f"    Genes: {adata.n_vars:,}")print(f"    .X:              Pearson residuals (UMI-corrected)")print(f"    layers['counts']: raw counts (untouched)")print(f"    layers['lognorm']: log-normalized expression")OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)print(f"\n  Saving to: {OUTPUT_FILE}")start_time = time.time()adata.write_h5ad(OUTPUT_FILE)elapsed = time.time() - start_timefile_size = OUTPUT_FILE.stat().st_size / 1e9print(f"\n✓ Saved in {elapsed:.1f}s ({file_size:.2f} GB)")

## Summary**Why.** Run record.<sub>source: `00.5_regressionCopy4.ipynb` cell 20</sub>

In [ ]:
why("Summary", "Run record")

In [ ]:
# ── source: 00.5_regressionCopy4.ipynb cell 20 ──print("\n" + "="*80)print("✓ MODULE 00.5 COMPLETE: UMI CORRECTION")print("="*80)print(f"""  Dataset: {DATASET}  Final dimensions:    Cells: {adata.n_obs:,}    Genes: {adata.n_vars:,}  Data structure:    adata.X:              Pearson residuals (UMI-corrected)    adata.layers['counts']: raw counts    adata.layers['lognorm']: log-normalized expression  UMI correction:    Before (log-norm):    ρ = {rho_pre:.4f}    After (Pearson):      ρ = {rho_post:.4f}    Clip value:           ±{CLIP_VALUE}  Output files:    Data:    {OUTPUT_FILE}    Figures: {FIGURES_DIR}/  → Ready for Module 01: Senescence Scoring""")

## GATE**Why.** Module 03 scores on `.X`. The three-layer contract must hold, and `.X` must actually be residuals (signed, unlike log-normalized counts which are >= 0).

In [ ]:
# ⟨NEW⟩ exit gate — layer contractwhy("02 GATE", "Does the layer contract hold for module 03?")import numpy as np, scipy.sparse as spassert_layers(adata, expect_X="Pearson residuals", require=('counts', 'lognorm'))_x = adata.X_mn = float(_x.data.min()) if sp.issparse(_x) else float(np.min(_x))_mx = float(_x.data.max()) if sp.issparse(_x) else float(np.max(_x))gate(".X has negative values (residuals, not log-norm)", _mn < 0, f"min={_mn:.2f}")gate(f".X clipped to +/-{C.CLIP_VALUE}", _mx <= C.CLIP_VALUE + 1e-6, f"max={_mx:.2f}")print(f"\n  depth coupling  before rho={rho_pre:+.4f}   after rho={rho_post:+.4f}")print("  NOTE: the decisive gate is score-vs-depth, measured in module 03.")